# Joint random forest (best Optuna config) + Breiman proximity prototypes
- **Data (v2):** the `tabular_data_prep` matrix and its GLOBAL selected feature set (`tabular_v2_*`), as a dense design = presence columns (0/1) + value columns (mean-imputed), at FULL record (no landmark censoring), and the SAME fixed 8-1-1 person split as ProtoPregNet (`protopregnet_landmark_cohort`).
- **Split use (mirrors ProtoPregNet):** train = fit the forest and extract prototypes; val = sanity metrics only (the forest has no early stopping; the Optuna study did use it to pick the hyperparameters); test = printed once for sanity, no decisions depend on it.
- **RF:** the BEST config of the most recent Optuna study (`rf_v2_hpo/best_params.json`, written by `mode=tune` of the RF v2 notebook): the same model as the full-record metrics cell. Prototypes are extracted post hoc from the trained forest's leaf co-membership (Breiman proximity) and assigned to REAL training patients, per class.
- **Two simple checks:** (1) do ProtoPregNet's prototype patients overlap with the RF's, and (2) do each RF prototype's features line up with its proximal neighbours (presence of concepts / lab windows).
- **Artifacts** are saved under `rf_v2_*` names, so the earlier forest and prototypes are not overwritten.

In [0]:
# # ============================================================================
# # FULL-RECORD performance of the Optuna-best RF config (self-contained: needs only Spark, the v2 tables / matrix cache, and best_params.json)
# #   STORED by mode=tune (best_params.json): the best trial's VALIDATION macro + per-label AUROC, the config and the data sizes. Nothing else.
# #   RECOMPUTED here: refit the best config on the full-record TRAIN rows (no landmark censoring, the global selected feature set), then the
# #   full metric suite on validation (reference) and on the held-out TEST split, with bootstrap CIs on test.
# # ============================================================================
# import os, json, time, shutil
# import numpy as np
# import pandas as pd
# import scipy.sparse as sp
# import sklearn
# from joblib import Parallel, delayed
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, brier_score_loss, balanced_accuracy_score)

# CLASS_NAMES = ["control", "pe_ghtn", "gdm"]
# OUTCOME_NAMES = ["pe_ghtn", "gdm"]
# LABEL_COLS = ["y_control", "y_pe_ghtn", "y_gdm"]


# def _pick_local_base():
#     for base in ("/local_disk0", "/tmp"):
#         if os.path.isdir(base) and os.access(base, os.W_OK):
#             return base
#     return "/tmp"


# CFG = {
#     "hpo_dir": "/path/to/exports/rf_v2_hpo",                       # best_params.json written by mode=tune of the RF v2 notebook
#     "tabular_meta_path": "/path/to/exports/tabular_v2/meta.json",
#     "selected_table": "catalog.schema.tabular_v2_selected_features",
#     "split_table": "catalog.schema.protopregnet_landmark_cohort",
#     "cache_dir": "/path/to/exports/tabular_v2_cache",
#     "local_cache_dir": f"{_pick_local_base()}/tabular_v2_cache",
#     "n_jobs": 32,                                                                      # forest fit / predict and the bootstrap
#     "n_bootstrap": 1000, "bootstrap_chunks": 8, "ci": 0.95, "bootstrap_seed": 42,     # same as the landmark runs
# }

# # ===== helpers: copied unchanged from the RF v2 notebook =====
# CACHE_FILES = ["X_data.npy", "X_indices.npy", "X_indptr.npy", "person_index.npy", "feature_names.json", "meta.json"]


# def _cp_from_volume(volume_path, local_path):
#     try:
#         dbutils.fs.cp(volume_path, f"file:{local_path}")
#     except Exception:
#         shutil.copyfile(volume_path, local_path)


# def read_meta(directory):
#     with open(f"{directory}/meta.json") as f:
#         return json.load(f)


# def ensure_local_cache():
#     """The matrix lives on the node's local disk so concurrent tasks can memory-map ONE copy (shared page cache). If the node has no
#     local copy (e.g. the cluster restarted), copy it from the Volume once; a task that loses the race just uses the winner's copy."""
#     loc = CFG["local_cache_dir"]
#     if os.path.exists(f"{loc}/meta.json"):
#         return loc
#     if not os.path.exists(f"{CFG['cache_dir']}/meta.json"):
#         raise FileNotFoundError(f"no matrix cache in {CFG['cache_dir']} -- run the notebook once with mode=prep first")
#     tmp = f"{loc}.tmp_{os.getpid()}"
#     shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
#     for fn in CACHE_FILES:
#         _cp_from_volume(f"{CFG['cache_dir']}/{fn}", f"{tmp}/{fn}")
#     try:
#         os.rename(tmp, loc)
#     except OSError:
#         shutil.rmtree(tmp, ignore_errors=True)                        # another task finished first
#     return loc


# def load_cache():
#     loc = ensure_local_cache()
#     with open(f"{loc}/meta.json") as f:
#         meta = json.load(f)
#     data, indices = np.load(f"{loc}/X_data.npy", mmap_mode="r"), np.load(f"{loc}/X_indices.npy", mmap_mode="r")
#     X = sp.csr_matrix((data, indices, np.load(f"{loc}/X_indptr.npy")), shape=tuple(meta["shape"]), copy=False)   # read-only, memory-mapped
#     with open(f"{loc}/feature_names.json") as f:
#         feature_names = json.load(f)
#     return X, feature_names, np.load(f"{loc}/person_index.npy")


# def matrix_fingerprint(feature_names, shape):
#     """Identifies one built matrix, so the selected-features table and every cache can be checked to belong together."""
#     import hashlib
#     h = hashlib.sha1(("|".join(feature_names) + f"#{shape[0]}x{shape[1]}").encode()).hexdigest()
#     return h[:16]


# def build_design_spec(sel, feature_names):
#     """sel: the selected rows of tabular_v2_selected_features. Restricts the stored matrix to the base columns that are needed and maps every design
#     column to its (restricted) base column."""
#     col_index = {n: i for i, n in enumerate(feature_names)}
#     missing = [c for c in sel["base_col"] if c not in col_index]
#     assert not missing, f"{len(missing)} selected base columns are not in the cached matrix, e.g. {missing[:3]}: rebuild the cache (mode=prep)"
#     base_full = sel["base_col"].map(col_index).to_numpy()
#     uniq = np.unique(base_full)
#     return {"uniq": uniq, "base_idx": np.searchsorted(uniq, base_full), "is_presence": (sel["role"] == "presence").to_numpy(),
#             "impute": sel["impute_value"].to_numpy(dtype=float), "design_cols": sel["design_col"].tolist()}


# def make_design_dense(Xc, base_idx, is_presence, impute):
#     """Dense float32 design for the SELECTED design columns, in the order given.
#       presence column: 1 if the stored base column has an entry (structure), else 0
#       value column:    the stored value if present, else impute (train mean of the present values)
#     Mean-imputation + the presence column is the standard 'value + indicator' encoding; trees are invariant to the constant, so no scaling or
#     centering is applied here."""
#     n = Xc.shape[0]
#     sub = Xc[:, np.asarray(base_idx)].tocoo()
#     out = np.tile(np.where(is_presence, 0.0, impute).astype(np.float32), (n, 1))
#     out[sub.row, sub.col] = np.where(is_presence[sub.col], 1.0, sub.data).astype(np.float32)
#     return out


# def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
#     results = {}
#     y_pred = (y_prob >= 0.5).astype(int)
#     for c, name in enumerate(class_names):
#         yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
#         n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
#         results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
#         results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
#         results[f"f1_{name}"] = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
#         results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
#         results[f"bacc_{name}"] = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

#     def macro(prefix):
#         vals = [results[f"{prefix}_{n}"] for n in class_names if not np.isnan(results[f"{prefix}_{n}"])]
#         return float(np.mean(vals)) if vals else float("nan")

#     for m in ["auroc", "auprc", "f1", "bacc", "brier"]:
#         results[f"{m}_macro"] = macro(m)
#     for m in ["auroc", "auprc"]:
#         vals = [results[f"{m}_{n}"] for n in outcome_names if not np.isnan(results[f"{m}_{n}"])]
#         results[f"{m}_macro_outcomes_only"] = float(np.mean(vals)) if vals else float("nan")
#     results["_primary"] = results["auroc_macro"]
#     return results


# def _boot_chunk(Y_true, Y_prob, class_names, n_iter, seed, keys):
#     rng = np.random.RandomState(seed)
#     n = Y_true.shape[0]
#     out = {k: [] for k in keys}
#     for _ in range(n_iter):
#         idx = rng.randint(0, n, n)
#         yt_r, yp_r = Y_true[idx], Y_prob[idx]
#         if any(yt_r[:, c].sum() == 0 or yt_r[:, c].sum() == n for c in range(len(class_names))):
#             continue
#         m = _compute_metric_suite(yt_r, yp_r, class_names=class_names, outcome_names=OUTCOME_NAMES)
#         for k in keys:
#             out[k].append(m[k])
#     return out


# def bootstrap_metrics_ci(Y_true, Y_prob, class_names):
#     """Same procedure as the ProtoPregNet landmark runs (resample test patients, recompute the suite, percentile CI), split into a FIXED
#     number of chunks with fixed seeds so the result does not depend on n_jobs; chunks run in parallel."""
#     stems = ["auroc", "auprc", "f1", "bacc", "brier"]
#     keys = ([f"{m}_macro" for m in stems] + ["auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]
#             + [f"{m}_{c}" for m in stems for c in class_names])
#     sizes = np.full(CFG["bootstrap_chunks"], CFG["n_bootstrap"] // CFG["bootstrap_chunks"])
#     sizes[: CFG["n_bootstrap"] % CFG["bootstrap_chunks"]] += 1
#     parts = Parallel(n_jobs=min(CFG["n_jobs"], CFG["bootstrap_chunks"]))(
#         delayed(_boot_chunk)(Y_true, Y_prob, class_names, int(s), CFG["bootstrap_seed"] + i, keys) for i, s in enumerate(sizes))
#     lo_q, hi_q = (1 - CFG["ci"]) / 2 * 100, (1 + CFG["ci"]) / 2 * 100
#     res = {}
#     for k in keys:
#         arr = np.concatenate([np.asarray(p[k], dtype=float) for p in parts]); arr = arr[~np.isnan(arr)]
#         res[k] = (float("nan"),) * 3 if len(arr) == 0 else (arr.mean(), *np.percentile(arr, [lo_q, hi_q]))
#     return res


# def fit_rf(Xtr, Ytr, hp, n_jobs):
#     if any(int(Ytr[:, i].sum()) in (0, len(Ytr)) for i in range(Ytr.shape[1])):
#         raise ValueError("a label has a single class in the training rows of this fit")
#     return RandomForestClassifier(**{**CFG["rf_fixed"], **hp, "n_jobs": int(n_jobs)}).fit(Xtr, Ytr)


# def predict_rf(model, Xd):
#     """(n, 3) probabilities, columns in CLASS_NAMES order."""
#     return np.column_stack([p[:, 1] for p in model.predict_proba(Xd)])


# # ===== 1. what is already stored =====
# path = f"{CFG['hpo_dir']}/best_params.json"
# if not os.path.exists(path):
#     raise FileNotFoundError(f"{path} not found: run mode=tune of the RF v2 notebook first")
# with open(path) as f:
#     BP = json.load(f)
# print(f"STORED in best_params.json: study '{BP['study_name']}', {BP['n_trials_complete']} complete / {BP['n_trials_pruned']} pruned trials (final={BP['final']})")
# print(f"  validation macro AUROC of the best trial {BP['best_value']:.4f} | by label: {({k: round(v, 4) for k, v in BP['val_auc_by_label'].items()})}")
# print(f"  trained on {BP['n_train']:,} train patients, validated on {BP['n_val']:,}, {BP['n_design_columns']:,} design columns, sklearn {BP.get('sklearn')} (now {sklearn.__version__})")
# print(f"  best params: {BP['best_params']} | fixed: {BP['rf_fixed']}")
# print("  NOT stored: test metrics, AUPRC / F1 / balanced accuracy / Brier, confidence intervals, the fitted model -> recomputed below\n")

# # ===== 2. data: the v2 matrix, the GLOBAL selected feature set, the ProtoPregNet split (same code path as the RF v2 notebook) =====
# t0 = time.time()
# X_full, feature_names, person_index = load_cache()
# pid_int_all = person_index.astype("int64")
# with open(CFG["tabular_meta_path"]) as f:
#     tmeta = json.load(f)
# FP = matrix_fingerprint(feature_names, X_full.shape)
# assert FP == tmeta["fingerprint"] == read_meta(CFG["local_cache_dir"]).get("fingerprint") == BP.get("matrix_fingerprint", FP), \
#     "cache / tabular_v2 meta.json / best_params.json disagree: the config was tuned on a different matrix"
# sel = (spark.table(CFG["selected_table"]).filter("selected_final = true")
#        .select("design_col", "role", "base_col", "impute_value", "matrix_fingerprint").toPandas())
# assert set(sel["matrix_fingerprint"]) == {FP}, "the selected-features table was built from a different matrix"
# sel = sel.sort_values(["role", "design_col"]).reset_index(drop=True)             # same column order as the notebook
# spec = build_design_spec(sel, feature_names)
# X = X_full[:, spec["uniq"]].tocsr()
# coh = spark.table(CFG["split_table"]).select("pid_int", "split", *LABEL_COLS).toPandas().set_index("pid_int")
# meta = coh.reindex(pid_int_all)
# assert meta["split"].notna().all(), "matrix patients missing from the split table"
# Y_all = meta[LABEL_COLS].to_numpy(dtype=int); split_all = meta["split"].to_numpy()
# rows = {s: np.flatnonzero(split_all == s) for s in ("train", "val", "test")}
# design = lambda r: make_design_dense(X[r], spec["base_idx"], spec["is_presence"], spec["impute"])      # FULL record: no censoring
# Xtr, Xva, Xte = design(rows["train"]), design(rows["val"]), design(rows["test"])
# Ytr, Yva, Yte = Y_all[rows["train"]], Y_all[rows["val"]], Y_all[rows["test"]]
# print(f"full-record design: train {Xtr.shape}, val {Xva.shape}, test {Xte.shape} | prevalence test (control, pe_ghtn, gdm) {Yte.mean(0).round(3).tolist()} | {time.time() - t0:.0f}s")

# # ===== 3. refit the best config on TRAIN (as in the study) and evaluate =====
# CFG["rf_fixed"] = BP["rf_fixed"]
# t0 = time.time()
# rf = fit_rf(Xtr, Ytr, BP["best_params"], CFG["n_jobs"])
# Pva, Pte = predict_rf(rf, Xva), predict_rf(rf, Xte)
# m_val, m_test = _compute_metric_suite(Yva, Pva), _compute_metric_suite(Yte, Pte)
# va_macro = float(np.mean([roc_auc_score(Yva[:, i], Pva[:, i]) for i in range(len(CLASS_NAMES))]))
# diff = abs(va_macro - BP["best_value"])
# print(f"refit in {time.time() - t0:.0f}s | recomputed validation macro AUROC {va_macro:.4f} vs stored {BP['best_value']:.4f} (|diff| {diff:.1e}): "
#       + ("reproduces the tuned model" if diff < 1e-6 else "DIFFERS - sklearn version, matrix or selected set may have changed since the study"))
# t0 = time.time(); ci = bootstrap_metrics_ci(Yte, Pte, CLASS_NAMES); print(f"bootstrap CIs ({CFG['n_bootstrap']} resamples) in {time.time() - t0:.0f}s")

# # ===== 4. result table: test (with 95% CI) and validation, same metric suite as the landmark runs =====
# out = []
# def add(scope, cls, stem, key):
#     _, lo, hi = ci[key]
#     out.append({"scope": scope, "class": cls, "metric": stem, "test": m_test[key], "test_ci_lo": lo, "test_ci_hi": hi, "val": m_val[key]})
# for stem in ["auroc", "auprc", "f1", "bacc", "brier"]:
#     add("macro", "ALL", stem, f"{stem}_macro")
#     for c in CLASS_NAMES:
#         add("class", c, stem, f"{stem}_{c}")
# for stem in ["auroc", "auprc"]:
#     add("macro_outcomes_only", "ALL", stem, f"{stem}_macro_outcomes_only")
# full_record_metrics = pd.DataFrame(out)
# pd.set_option("display.width", 200)
# print(f"\nFULL-RECORD performance of the best RF config (fit on {len(rows['train']):,} train patients; test n = {len(rows['test']):,}; F1 / balanced accuracy / Brier at threshold 0.5)")
# print(full_record_metrics.round(4).to_string(index=False))
# full_record_metrics

In [0]:
import os, time, json, shutil, pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             brier_score_loss, balanced_accuracy_score)

PERSON_ID_COL = "person_id"
CLASS_NAMES = ["control", "pe_ghtn", "gdm"]
OUTCOME_NAMES = ["pe_ghtn", "gdm"]
LABEL_COLS = ["y_control", "y_pe_ghtn", "y_gdm"]          # same label definitions as the landmark cohort table


def _pick_local_base():
    for base in ("/local_disk0", "/tmp"):
        if os.path.isdir(base) and os.access(base, os.W_OK):
            return base
    return "/tmp"


CFG = {
    # inputs: written by tabular_data_prep, and the ProtoPregNet split
    "selected_table": "catalog.schema.tabular_v2_selected_features",         # the GLOBAL selected feature set (selected_final = true)
    "tabular_meta_path": "/path/to/exports/tabular_v2/meta.json",
    "split_table": "catalog.schema.protopregnet_landmark_cohort",            # fixed 8-1-1 split + labels (seed 42)
    "cache_dir": "/path/to/exports/tabular_v2_cache",          # the v2 matrix cache shared with the RF / XGBoost v2 notebooks
    "local_cache_dir": f"{_pick_local_base()}/tabular_v2_cache",

    # the forest: the BEST config of the most recent Optuna study (full-record data, train -> val objective)
    "best_params_path": "/path/to/exports/rf_v2_hpo/best_params.json",

    # data / model
    "fit_on": "train",            # "train" (the data the study fit on, and ProtoPregNet's) | "train+val" (90%)
    "n_jobs": 8,                  # speed only (fit / apply); results do not depend on it when random_state is fixed

    # prototypes (Breiman, post hoc)
    "n_prototypes": {"control": 17, "pe_ghtn": 9, "gdm": 9},   # X per class; defaults = ProtoPregNet's GLOBAL-branch counts
    "k_neighbors": 20,            # cluster = center + its k nearest by proximity; larger k -> broader clusters
    "neighbor_pool_mult": 5,      # keep top (mult * k) neighbours per patient so removing clusters can't exhaust a list
    "assign_to": "medoid",        # real patient standing in for a cluster: "medoid" | "center" (see cell 9 docstring)
    "prefer_pure": True,          # prefer single-label patients as the representative (as in ProtoPregNet's projection)
    "block_rows": 1000,           # memory knob for the neighbour search, not a result knob

    # cross-model comparison: name -> ProtoPregNet export pickle (needs global_/local_projection_map)
    "ppn_exports": {"original": "/path/to/exports/full_analysis_export.pkl"},
    "seed": 42,

    # outputs
    "export_dir": "/path/to/exports",
    "artifact_prefix": "rf_v2",   # artifacts are saved as <prefix>_joint_<fit_on>.joblib / <prefix>_prototypes_<fit_on>.pkl: the earlier rf_joint_* files are not overwritten
    "save_artifacts": True,
}

# ---- the forest's config: fixed settings + the Optuna-best hyperparameters, exactly as the study used them ----
with open(CFG["best_params_path"]) as f:
    BP = json.load(f)
RF_PARAMS = {**BP["rf_fixed"], **BP["best_params"]}
print(f"best config from study '{BP['study_name']}' ({BP['n_trials_complete']} complete trials, validation macro AUROC {BP['best_value']:.4f}): {RF_PARAMS}")
if BP.get("sklearn") != sklearn.__version__:
    print(f"NOTE: the study ran on scikit-learn {BP.get('sklearn')}, this session has {sklearn.__version__}; the forest may differ slightly (do not pip-upgrade scikit-learn here)")


In [0]:
# ===== v2 matrix cache I/O + design builders: copied unchanged from the RF v2 notebook =====
CACHE_FILES = ["X_data.npy", "X_indices.npy", "X_indptr.npy", "person_index.npy", "feature_names.json", "meta.json"]


def _cp_from_volume(volume_path, local_path):
    try:
        dbutils.fs.cp(volume_path, f"file:{local_path}")
    except Exception:
        shutil.copyfile(volume_path, local_path)


def read_meta(directory):
    with open(f"{directory}/meta.json") as f:
        return json.load(f)


def ensure_local_cache():
    """The matrix lives on the node's local disk so concurrent tasks can memory-map ONE copy (shared page cache). If the node has no
    local copy (e.g. the cluster restarted), copy it from the Volume once; a task that loses the race just uses the winner's copy."""
    loc = CFG["local_cache_dir"]
    if os.path.exists(f"{loc}/meta.json"):
        return loc
    if not os.path.exists(f"{CFG['cache_dir']}/meta.json"):
        raise FileNotFoundError(f"no matrix cache in {CFG['cache_dir']} -- run the notebook once with mode=prep first")
    tmp = f"{loc}.tmp_{os.getpid()}"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    for fn in CACHE_FILES:
        _cp_from_volume(f"{CFG['cache_dir']}/{fn}", f"{tmp}/{fn}")
    try:
        os.rename(tmp, loc)
    except OSError:
        shutil.rmtree(tmp, ignore_errors=True)                        # another task finished first
    return loc


def load_cache():
    loc = ensure_local_cache()
    with open(f"{loc}/meta.json") as f:
        meta = json.load(f)
    data, indices = np.load(f"{loc}/X_data.npy", mmap_mode="r"), np.load(f"{loc}/X_indices.npy", mmap_mode="r")
    X = sp.csr_matrix((data, indices, np.load(f"{loc}/X_indptr.npy")), shape=tuple(meta["shape"]), copy=False)   # read-only, memory-mapped
    with open(f"{loc}/feature_names.json") as f:
        feature_names = json.load(f)
    return X, feature_names, np.load(f"{loc}/person_index.npy")


def matrix_fingerprint(feature_names, shape):
    """Identifies one built matrix, so the selected-features table and every cache can be checked to belong together."""
    import hashlib
    h = hashlib.sha1(("|".join(feature_names) + f"#{shape[0]}x{shape[1]}").encode()).hexdigest()
    return h[:16]


def build_design_spec(sel, feature_names):
    """sel: the selected rows of tabular_v2_selected_features. Restricts the stored matrix to the base columns that are needed and maps every design
    column to its (restricted) base column."""
    col_index = {n: i for i, n in enumerate(feature_names)}
    missing = [c for c in sel["base_col"] if c not in col_index]
    assert not missing, f"{len(missing)} selected base columns are not in the cached matrix, e.g. {missing[:3]}: rebuild the cache (mode=prep)"
    base_full = sel["base_col"].map(col_index).to_numpy()
    uniq = np.unique(base_full)
    return {"uniq": uniq, "base_idx": np.searchsorted(uniq, base_full), "is_presence": (sel["role"] == "presence").to_numpy(),
            "impute": sel["impute_value"].to_numpy(dtype=float), "design_cols": sel["design_col"].tolist()}


def make_design_dense(Xc, base_idx, is_presence, impute):
    """Dense float32 design for the SELECTED design columns, in the order given.
      presence column: 1 if the stored base column has an entry (structure), else 0
      value column:    the stored value if present, else impute (train mean of the present values)
    Mean-imputation + the presence column is the standard 'value + indicator' encoding; trees are invariant to the constant, so no scaling or
    centering is applied here."""
    n = Xc.shape[0]
    sub = Xc[:, np.asarray(base_idx)].tocoo()
    out = np.tile(np.where(is_presence, 0.0, impute).astype(np.float32), (n, 1))
    out[sub.row, sub.col] = np.where(is_presence[sub.col], 1.0, sub.data).astype(np.float32)
    return out

In [0]:
# ============================================================================
# Load the v2 matrix + the GLOBAL selected feature set + the fixed split/labels, align rows
# ============================================================================
X_full, feature_names, person_index = load_cache()
pid_int_all = pd.Index(person_index.astype("int64"))
with open(CFG["tabular_meta_path"]) as f:
    tmeta = json.load(f)
FP = matrix_fingerprint(feature_names, X_full.shape)
assert FP == tmeta["fingerprint"] == read_meta(CFG["local_cache_dir"]).get("fingerprint") == BP.get("matrix_fingerprint", FP), \
    "cache / tabular_v2 meta.json / best_params.json disagree: the config was tuned on a different matrix"

sel = (spark.table(CFG["selected_table"]).filter("selected_final = true")
       .select("design_col", "role", "group", "stat", "base_col", "impute_value", "concept_id", "window_wk", "concept_name", "matrix_fingerprint").toPandas())
assert len(sel) > 0 and set(sel["matrix_fingerprint"]) == {FP}, "the selected-features table is empty or was built from a different matrix"
sel = sel.sort_values(["role", "design_col"]).reset_index(drop=True)             # same column order as the RF v2 notebook
spec = build_design_spec(sel, feature_names)
X = X_full[:, spec["uniq"]].tocsr()                                              # only the stored columns the design needs
design = lambda rows: make_design_dense(X[rows], spec["base_idx"], spec["is_presence"], spec["impute"])      # FULL record: no censoring

coh = spark.table(CFG["split_table"]).select("pid_int", "split", *LABEL_COLS).toPandas().set_index("pid_int")
meta = coh.reindex(pid_int_all)
assert meta["split"].notna().all(), f"{int(meta['split'].isna().sum())} matrix patients are missing from {CFG['split_table']}"
Y_all = meta[LABEL_COLS].to_numpy(dtype=int)
split_all = meta["split"].to_numpy()
pids_all = np.asarray(pid_int_all).astype(str)
print(f"matrix {X_full.shape} -> {len(sel):,} design columns ({int(spec['is_presence'].sum()):,} presence + {int((~spec['is_presence']).sum()):,} value) | split sizes: {pd.Series(split_all).value_counts().to_dict()}")


In [0]:
# ============================================================================
# RF fit set and the dense design matrices (the feature set is the global selected set: no per-notebook selection)
# ============================================================================
fit_splits = ["train"] if CFG["fit_on"] == "train" else ["train", "val"]
fit_mask = np.isin(split_all, fit_splits)
val_mask, test_mask = split_all == "val", split_all == "test"
if CFG["fit_on"] != "train":
    print("NOTE: val is part of the fit set, so val metrics below are in-sample")

Y_train, pids_train = Y_all[fit_mask], pids_all[fit_mask]                          # "train" below = the RF fit set
X_train_sel = design(np.flatnonzero(fit_mask))
X_val_sel, X_test_sel = design(np.flatnonzero(val_mask)), design(np.flatnonzero(test_mask))
Y_val, Y_test = Y_all[val_mask], Y_all[test_mask]
X_all_sel = sp.csr_matrix(design(np.arange(len(split_all))))                       # sparse, any patient in matrix order (used by the cross-model cell)
FEATURE_NAMES = np.asarray(spec["design_cols"])                                    # design column names: 'has::<group>' (presence) and the stored value columns
selected_idx = np.arange(len(FEATURE_NAMES))                                       # kept for the saved artifact: positions in the design

# PRESENCE view, for the inspection / alignment cells. 'Present' = the concept / lab window has an entry in the record. It is read from the presence
# columns, not from `value != 0` (value columns are mean-imputed, so they are non-zero for everyone).
IS_PRESENCE = spec["is_presence"]
PRES_IDX = np.flatnonzero(IS_PRESENCE)
PRES_NAMES = FEATURE_NAMES[PRES_IDX]
P_TRAIN = X_train_sel[:, PRES_IDX] > 0                                             # (n_fit, n_presence) bool
_val_cols = sel[(sel["role"] == "value") & sel["stat"].isin(["week", "min"])].drop_duplicates("group").reset_index().set_index("group")["index"]
PRES_VALUE_COL = sel.loc[IS_PRESENCE, "group"].map(_val_cols).fillna(-1).astype(int).to_numpy()   # the value column paired with each presence column: event week / lab window min (or -1)
print(f"X_train_sel {X_train_sel.shape} ({X_train_sel.nbytes/1e9:.2f} GB), val {X_val_sel.shape}, test {X_test_sel.shape} | presence columns {len(PRES_IDX):,}")


In [0]:
# (removed) `!pip install --upgrade scikit-learn`: the Optuna study ran on the cluster's scikit-learn (its version is stored in best_params.json and checked in cell 2).
# Upgrading it here could change the forest, and then these prototypes would not belong to the tuned model.

In [0]:
# ============================================================================
# Fit the joint RF (Optuna-best config) and print sanity metrics
# ============================================================================
# ============================================================================
# Metric suite (unchanged) + fast macro AUROC for per-epoch early stopping
# ============================================================================
def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
    results = {}
    y_pred = (y_prob >= 0.5).astype(int)
    for c, name in enumerate(class_names):
        yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
        n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
        results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"f1_{name}"] = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
        results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
        results[f"bacc_{name}"] = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

    def macro(prefix):
        vals = [results[f"{prefix}_{n}"] for n in class_names if not np.isnan(results[f"{prefix}_{n}"])]
        return float(np.mean(vals)) if vals else float("nan")

    for m in ["auroc", "auprc", "f1", "bacc", "brier"]:
        results[f"{m}_macro"] = macro(m)
    for m in ["auroc", "auprc"]:
        vals = [results[f"{m}_{n}"] for n in outcome_names if not np.isnan(results[f"{m}_{n}"])]
        results[f"{m}_macro_outcomes_only"] = float(np.mean(vals)) if vals else float("nan")
    results["_primary"] = results["auroc_macro"]
    return results


def macro_auroc(y_true, y_prob):
    """== _compute_metric_suite(...)['auroc_macro'] but computes only that (early stopping runs it every epoch)."""
    vals = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        n_pos = int(yt.sum())
        if n_pos > 0 and n_pos < len(yt):
            vals.append(roc_auc_score(yt, y_prob[:, c]))
    return float(np.mean(vals)) if vals else float("nan")

t0 = time.time()
proto_rf = RandomForestClassifier(**{**RF_PARAMS, "n_jobs": CFG["n_jobs"]}).fit(X_train_sel, Y_train)       # the Optuna-best config
print(f"fitted in {time.time() - t0:.0f}s ({RF_PARAMS['n_estimators']} trees, bootstrap={RF_PARAMS['bootstrap']})")

def predict_matrix(model, Xd):
    pl = model.predict_proba(Xd)                       # list of (n, 2), one per output
    return np.column_stack([pl[i][:, 1] for i in range(len(CLASS_NAMES))])

for nm, Xd, Yd in [("val", X_val_sel, Y_val), ("test", X_test_sel, Y_test)]:
    m = _compute_metric_suite(Yd, predict_matrix(proto_rf, Xd))
    print(f"{nm}: macro AUROC {m['auroc_macro']:.4f} AUPRC {m['auprc_macro']:.4f} F1 {m['f1_macro']:.4f} "
          f"BalAcc {m['bacc_macro']:.4f} Brier {m['brier_macro']:.4f} | AUROC "
          + " ".join(f"{c} {m[f'auroc_{c}']:.4f}" for c in CLASS_NAMES))
# reproduction check: with the same data and config, validation macro AUROC must equal the study's best trial
if CFG["fit_on"] == "train":
    va_macro = macro_auroc(Y_val, predict_matrix(proto_rf, X_val_sel)); diff = abs(va_macro - BP["best_value"])
    print(f"validation macro AUROC {va_macro:.4f} vs the study's best trial {BP['best_value']:.4f} (|diff| {diff:.1e}): "
          + ("this is the tuned model" if diff < 1e-6 else "DIFFERS - scikit-learn version, matrix or selected set may have changed since the study"))


In [0]:
# ============================================================================
# Breiman proximity from leaf co-membership + top-K neighbours for every fit-set patient
# ============================================================================
# proximity(i, j) = fraction of trees in which i and j land in the SAME leaf. Build a sparse one-hot patient x leaf
# matrix L (one 1 per tree per patient); proximity = L @ L.T / n_trees without ever forming an n x n matrix.
# All 3 outputs share the same trees (multi-output RF), so this is ONE joint proximity for all classes.
n_train = X_train_sel.shape[0]
leaves = proto_rf.apply(X_train_sel)                                          # (n_train, n_trees) leaf id per tree
node_counts = np.array([e.tree_.node_count for e in proto_rf.estimators_])
offsets = np.concatenate([[0], np.cumsum(node_counts)[:-1]])                  # make node ids unique across trees
n_trees = leaves.shape[1]
rows = np.repeat(np.arange(n_train), n_trees)                                 # patient index per (patient, tree) pair
cols = (leaves + offsets[None, :]).ravel()                                    # global leaf column per pair (row-major)
L = sp.csr_matrix((np.ones(rows.size, dtype=np.float32), (rows, cols)), shape=(n_train, int(node_counts.sum())))
print(f"{n_train} patients | {n_trees} trees | {int(node_counts.sum())} nodes | leaf matrix nnz {L.nnz}")

def top_neighbors(L, n_trees, K, block_rows):
    """For each patient: the K other patients with the highest proximity, sorted descending.
    Row-block sparse product L[block] @ L.T keeps memory at block_rows x n_train floats."""
    n = L.shape[0]
    Lt = L.T.tocsr()
    nb_idx = np.empty((n, K), dtype=np.int32)
    nb_prox = np.empty((n, K), dtype=np.float32)
    for s in range(0, n, block_rows):
        e = min(s + block_rows, n)
        P = (L[s:e] @ Lt).toarray() / n_trees                     # (block, n) proximities in [0, 1]
        P[np.arange(e - s), np.arange(s, e)] = -1.0               # a patient is not its own neighbour
        part = np.argpartition(-P, K, axis=1)[:, :K]              # unordered top-K per row (vectorized)
        pv = np.take_along_axis(P, part, axis=1)
        order = np.argsort(-pv, axis=1)
        nb_idx[s:e] = np.take_along_axis(part, order, axis=1)
        nb_prox[s:e] = np.take_along_axis(pv, order, axis=1)
    return nb_idx, nb_prox

K_POOL = CFG["k_neighbors"] * CFG["neighbor_pool_mult"]
t0 = time.time()
nb_idx, nb_prox = top_neighbors(L, n_trees, K_POOL, CFG["block_rows"])
print(f"neighbour search {time.time() - t0:.0f}s | median proximity of nearest neighbour {np.median(nb_prox[:, 0]):.3f}, "
      f"of the k-th {np.median(nb_prox[:, CFG['k_neighbors'] - 1]):.3f}")

In [0]:
# ============================================================================
# Breiman's prototype algorithm per class, then assign each cluster to a REAL training patient
# ============================================================================
def extract_prototypes(nb_idx, nb_prox, leaves, Y, pids, class_names, cfg):
    """Per class c, repeat X times (Breiman):
       1. among still-available class-c patients, pick the one whose k nearest still-available neighbours (over ALL
          patients) most often also carry label c ("surrounded by the same class"); ties -> higher mean proximity.
          This is the CENTER.
       2. cluster = center + its k neighbours.
       3. the prototype is a REAL patient standing in for the cluster instead of median feature values:
            assign_to="center": the center itself (the cluster's hub: every member is among ITS k nearest, but members
                                need not be close to each other);
            assign_to="medoid": the member with the highest mean proximity to the rest of the cluster (the most
                                typical member; the analogue of 'nearest real patient to the median profile').
          Both are restricted to class-c members not already used by another prototype (single-label preferred).
       4. remove the cluster and repeat.
    Cluster removal is per class; real patients are unique ACROSS classes (cross-class exclusion, as in ProtoPregNet's
    projection). Multi-label: 'class c' means label c == 1."""
    n, k = len(Y), cfg["k_neighbors"]
    single_label = Y.sum(axis=1) == 1
    claimed = np.zeros(n, dtype=bool)
    records, groups = [], {}
    for ci, cname in enumerate(class_names):
        want = cfg["n_prototypes"].get(cname, 0)
        yc = Y[:, ci] == 1
        alive = np.ones(n, dtype=bool)
        found, attempts = 0, 0
        while found < want and attempts < 3 * want:
            attempts += 1
            alive_nb = alive[nb_idx]                                  # (n, K_pool): is each listed neighbour still available?
            first_k = alive_nb & (np.cumsum(alive_nb, axis=1) <= k)   # each patient's k nearest AVAILABLE neighbours
            n_k = first_k.sum(axis=1)
            cand = alive & yc & (n_k >= k)                            # never score a patient on fewer than k neighbours
            if not cand.any():
                print(f"  WARNING: class '{cname}': no candidates left after {found}/{want} prototypes")
                break
            purity = (yc[nb_idx] & first_k).sum(axis=1) / np.maximum(n_k, 1)
            mean_prox = (nb_prox * first_k).sum(axis=1) / np.maximum(n_k, 1)
            center = int(np.argmax(np.where(cand, purity + 1e-3 * mean_prox, -1.0)))    # 1e-3*prox < 1/k: pure tie-break
            members = np.concatenate([[center], nb_idx[center][first_k[center]]])
            alive[members] = False                                    # remove the cluster whether or not it yields a prototype

            lv = leaves[members]                                      # pairwise proximities inside the cluster
            prox_mm = (lv[:, None, :] == lv[None, :, :]).mean(axis=2)
            np.fill_diagonal(prox_mm, 0.0)
            mean_to_group = prox_mm.sum(axis=1) / (len(members) - 1)

            ok = yc[members] & ~claimed[members]
            if not ok.any():
                print(f"  WARNING: class '{cname}': cluster had no unclaimed class-{cname} member; skipped")
                continue
            used_fallback = False
            if cfg["prefer_pure"]:
                pure_ok = ok & single_label[members]
                used_fallback = not pure_ok.any()
                ok = pure_ok if pure_ok.any() else ok
            if cfg["assign_to"] == "center" and ok[0]:
                pos = 0
            else:
                pos = int(np.flatnonzero(ok)[np.argmax(mean_to_group[ok])])
            rep = int(members[pos])
            claimed[rep] = True
            pidx = len(records)
            groups[pidx] = members
            records.append({
                "proto_idx": pidx, "class": cname, "train_idx": rep, "patient_id": str(pids[rep]),
                "center_idx": int(center), "rep_is_center": rep == int(center), "group_size": len(members),
                "center_neighbor_purity": float(purity[center]),
                **{f"group_frac_{cn}": float(Y[members, j].mean()) for j, cn in enumerate(class_names)},
                "mean_within_group_prox": float(prox_mm.sum() / (len(members) * (len(members) - 1))),
                "rep_mean_prox_to_group": float(mean_to_group[pos]),
                "rep_single_label": bool(single_label[rep]), "used_multilabel_fallback": used_fallback,
            })
            found += 1
        print(f"class '{cname}': {found}/{want} prototypes")
    return pd.DataFrame(records), groups


proto_df, PROTO_GROUPS = extract_prototypes(nb_idx, nb_prox, leaves, Y_train, pids_train, CLASS_NAMES, CFG)
print(f"\n{len(proto_df)} prototypes; clusters cover {sum(len(g) for g in PROTO_GROUPS.values())} patient slots "
      f"({len(np.unique(np.concatenate(list(PROTO_GROUPS.values()))))} unique) of {n_train} fit-set patients")
proto_df.round(3)

In [0]:
# ============================================================================
# Inspect a prototype: how its cluster's concepts / lab windows differ from the rest of its class (PRESENCE rates)
# ============================================================================
CFG_INSPECT = {"top_features": 15, "min_group_rate": 0.0}
_presence_cache = {}

def presence_rate(mask_key, mask):
    """Share of patients with each concept / lab window PRESENT (read from the presence columns of the design)."""
    if mask_key not in _presence_cache:
        _presence_cache[mask_key] = P_TRAIN[mask].mean(axis=0)
    return _presence_cache[mask_key]

def proto_profile(proto_idx, top=None):
    top = top or CFG_INSPECT["top_features"]
    row = proto_df.loc[proto_df["proto_idx"] == proto_idx].iloc[0]
    ci = CLASS_NAMES.index(row["class"])
    members = PROTO_GROUPS[proto_idx]
    Pg = P_TRAIN[members]
    g_rate = Pg.mean(axis=0)
    c_rate = presence_rate(("class", ci), Y_train[:, ci] == 1)
    all_rate = presence_rate(("all",), np.ones(len(P_TRAIN), dtype=bool))
    diff = g_rate - c_rate
    order = np.argsort(-np.abs(diff))
    order = order[g_rate[order] >= CFG_INSPECT["min_group_rate"]][:top]
    vcol = PRES_VALUE_COL[order]                                                   # paired value column: event first-occurrence week / lab window min (-1 = none selected)
    Xg = X_train_sel[members]
    med = [np.median(Xg[Pg[:, j], v]) if v >= 0 and Pg[:, j].any() else np.nan for j, v in zip(order, vcol)]
    rep = int(row["train_idx"])
    rep_val = [X_train_sel[rep, v] if v >= 0 and P_TRAIN[rep, j] else np.nan for j, v in zip(order, vcol)]
    return pd.DataFrame({
        "feature": PRES_NAMES[order], "group_rate": g_rate[order], f"{row['class']}_rate": c_rate[order],
        "overall_rate": all_rate[order], "diff_vs_class": diff[order], "group_median_value": med,
        "representative_value": rep_val})

PROTO_TO_INSPECT = 26
print(proto_df.loc[proto_df["proto_idx"] == PROTO_TO_INSPECT].T)
proto_profile(PROTO_TO_INSPECT).round(3)


In [0]:
# ============================================================================
# CHECK 1 -- do ProtoPregNet's prototype patients overlap with the RF's?
#   (a) same patient as an RF representative; (b) inside the cluster of an RF prototype (center + its k neighbours),
#   compared with how many would land there by chance if ProtoPregNet's prototypes were random patients of that class.
# ============================================================================
fit_row_of_pid = pd.Series(np.arange(n_train), index=pids_train)           # patient id -> row in the RF fit set
rep_idx = proto_df["train_idx"].to_numpy()
rep_proto_of_row = np.full(n_train, -1, dtype=int)                          # row -> RF prototype it represents (-1 = none)
rep_proto_of_row[rep_idx] = proto_df["proto_idx"].to_numpy()
cluster_proto = {c: np.full(n_train, -1, dtype=int) for c in CLASS_NAMES}   # row -> RF prototype of class c whose cluster holds it
for p, members in PROTO_GROUPS.items():                                     # clusters are disjoint within a class
    cluster_proto[proto_df.loc[proto_df["proto_idx"] == p, "class"].iloc[0]][members] = p

class _NumpyCompatUnpickler(pickle.Unpickler):
    """Loads pickles written under NumPy 2.x on NumPy 1.x (numpy._core is the renamed numpy.core)."""
    def find_class(self, module, name):
        if module == "numpy._core" or module.startswith("numpy._core."):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)


def load_pickle_compat(path):
    with open(path, "rb") as f:
        return _NumpyCompatUnpickler(f).load()


def load_ppn_prototypes(path):
    """ProtoPregNet prototypes (one real patient each) from an export pickle, per branch. Works for the full exports
    (original, landmark, washout) and for slim re-exports; branches missing from the pickle are skipped."""
    ex = load_pickle_compat(path)
    return {b: pd.DataFrame([{"ppn_proto_idx": p, "patient_id": str(v["patient_id"]), "class": v["class_name"]}
                             for p, v in ex[f"{b}_projection_map"].items()])
            for b in ("global", "local") if f"{b}_projection_map" in ex}

def overlap_with_rf(ppn, name):
    fr = fit_row_of_pid.reindex(ppn["patient_id"]).to_numpy()
    in_fit = ~np.isnan(fr)
    fr = np.where(in_fit, fr, 0).astype(int)
    cls = ppn["class"].to_numpy()
    same_class_cluster = np.full(len(ppn), -1, dtype=int)
    for c in CLASS_NAMES:
        m = cls == c
        same_class_cluster[m] = cluster_proto[c][fr[m]]
    same_class_cluster[~in_fit] = -1
    any_cluster = in_fit & np.any([cluster_proto[c][fr] >= 0 for c in CLASS_NAMES], axis=0)
    per = pd.DataFrame({"export": name, "ppn_proto_idx": ppn["ppn_proto_idx"], "class": cls, "ppn_patient_id": ppn["patient_id"],
                        "in_rf_fit_set": in_fit,
                        "same_patient_as_rf_proto": np.where(in_fit, rep_proto_of_row[fr], -1),
                        "in_cluster_of_rf_proto": same_class_cluster, "in_any_rf_cluster": any_cluster})
    rows = []
    for ci, c in enumerate(CLASS_NAMES):
        s_ = per[per["class"] == c]
        coverage = float(((cluster_proto[c] >= 0) & (Y_train[:, ci] == 1)).sum() / (Y_train[:, ci] == 1).sum())
        rows.append({"export": name, "class": c, "n_ppn": len(s_), "n_in_rf_fit_set": int(s_["in_rf_fit_set"].sum()),
                     "same_patient": int((s_["same_patient_as_rf_proto"] >= 0).sum()),
                     "in_same_class_rf_cluster": int((s_["in_cluster_of_rf_proto"] >= 0).sum()),
                     "expected_by_chance": round(float(s_["in_rf_fit_set"].sum()) * coverage, 2),
                     "rf_cluster_coverage_of_class": round(coverage, 4)})
    return per, pd.DataFrame(rows)


overlap_pairs, overlap_summary = [], []
for ex_name, path in CFG["ppn_exports"].items():
    for branch, ppn in load_ppn_prototypes(path).items():
        per, summ = overlap_with_rf(ppn, f"{ex_name}/{branch}")
        overlap_pairs.append(per); overlap_summary.append(summ)
if overlap_pairs:
    overlap_pairs, overlap_summary = pd.concat(overlap_pairs, ignore_index=True), pd.concat(overlap_summary, ignore_index=True)
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
    print(overlap_summary.to_string(index=False))
    print("\nProtoPregNet prototypes that overlap (same patient, or inside an RF cluster of their class):")
    print(overlap_pairs[(overlap_pairs["same_patient_as_rf_proto"] >= 0) | (overlap_pairs["in_cluster_of_rf_proto"] >= 0)]
          .drop(columns=["in_rf_fit_set"]).to_string(index=False))
else:
    overlap_pairs, overlap_summary = pd.DataFrame(), pd.DataFrame()
    print("no ProtoPregNet exports configured in CFG['ppn_exports'] -- nothing to compare")

In [0]:
# ============================================================================
# CHECK 2 -- do an RF prototype's features line up with its proximal neighbours?
#   Neighbours = the prototype patient's own k nearest fit-set patients by RF proximity (any class).
#   "present" = the concept / lab window is present in the record (the design's presence columns). Two numbers per prototype, each against a NULL of k random
#   same-class patients (averaged over several draws):
#     jaccard   = mean overlap of present-feature sets between the prototype and each neighbour
#     precision = of the prototype's present features, the average share of neighbours that also have them
# ============================================================================
CFG_ALIGN = {"k": CFG["k_neighbors"], "null_draws": 20}
rng_a = np.random.default_rng(CFG["seed"])


def alignment_stats(a, N):
    """a: (F,) bool features present in the prototype patient; N: (..., m, F) bool comparison patients."""
    inter, union = (N & a).sum(axis=-1), (N | a).sum(axis=-1)
    jac = np.where(union == 0, 1.0, inter / np.maximum(union, 1)).mean(axis=-1)     # two empty profiles count as identical
    prec = N[..., a].mean(axis=-1).mean(axis=-1) if a.any() else np.full(jac.shape, np.nan)     # undefined for an empty profile (n_present = 0)
    return jac, prec


rows = []
for _, r in proto_df.iterrows():
    ci, rep = CLASS_NAMES.index(r["class"]), int(r["train_idx"])
    nbrs = nb_idx[rep, :CFG_ALIGN["k"]]
    a = P_TRAIN[rep]
    jac, prec = alignment_stats(a, P_TRAIN[nbrs])
    pool = np.setdiff1d(np.flatnonzero(Y_train[:, ci] == 1), np.concatenate([[rep], nbrs]))
    null_rows = rng_a.choice(pool, size=(CFG_ALIGN["null_draws"], CFG_ALIGN["k"]), replace=True)
    njac, nprec = alignment_stats(a, P_TRAIN[null_rows])
    rows.append({"proto_idx": r["proto_idx"], "class": r["class"], "patient_id": r["patient_id"], "n_present": int(a.sum()),
                 "nbr_jaccard": float(jac), "null_jaccard": float(njac.mean()), "jaccard_lift": float(jac / max(njac.mean(), 1e-9)),
                 "nbr_precision": float(prec), "null_precision": float(nprec.mean()),
                 "precision_lift": float(prec / max(nprec.mean(), 1e-9)),
                 "nbr_same_class_share": float(Y_train[nbrs, ci].mean())})
alignment_df = pd.DataFrame(rows)
alignment_summary = alignment_df.groupby("class")[["n_present", "nbr_jaccard", "null_jaccard", "jaccard_lift", "nbr_precision",
                                                   "null_precision", "precision_lift", "nbr_same_class_share"]].mean()
alignment_summary["class_prevalence"] = [Y_train[:, CLASS_NAMES.index(c)].mean() for c in alignment_summary.index]
print(alignment_summary.round(3).to_string())
alignment_df.round(3)

In [0]:
# ============================================================================
# Save artifacts so the same forest/prototypes can be reused (e.g. for time-dependent evaluation)
# ============================================================================
if CFG["save_artifacts"]:
    tag = CFG["fit_on"].replace("+", "_")
    rf_path = f"{CFG['export_dir']}/{CFG['artifact_prefix']}_joint_{tag}.joblib"
    joblib.dump(proto_rf, rf_path, compress=3)
    proto_path = f"{CFG['export_dir']}/{CFG['artifact_prefix']}_prototypes_{tag}.pkl"
    with open(proto_path, "wb") as f:
        pickle.dump({"proto_df": proto_df,
                     "groups_patient_ids": {p: pids_train[m] for p, m in PROTO_GROUPS.items()},
                     "selected_feature_names": FEATURE_NAMES, "selected_idx": selected_idx, "design_columns": sel,
                     "cfg": {k: v for k, v in CFG.items()}, "rf_params": RF_PARAMS, "best_params_json": BP, "matrix_fingerprint": FP,
                     "overlap_summary": overlap_summary, "overlap_per_prototype": overlap_pairs,
                     "alignment_per_prototype": alignment_df, "alignment_summary": alignment_summary}, f)
    print(f"saved {rf_path} ({os.path.getsize(rf_path)/1e6:.0f} MB) and {proto_path}")


In [0]:
# ============================================================================
# CHECK 1b -- GRADED overlap: how close is each ProtoPregNet prototype to its nearest SAME-CLASS RF prototype,
# compared with random same-class training patients? (proximity = fraction of trees sharing a leaf)
# ============================================================================
CFG_GRADED = {"n_null": 300, "seed": CFG["seed"]}
row_of_pid = pd.Series(np.arange(len(pids_all)), index=pids_all)           # patient id -> row in the full matrix
rep_idx = proto_df["train_idx"].to_numpy()
rep_cls = proto_df["class"].to_numpy()
rep_proto = proto_df["proto_idx"].to_numpy()

# NULL: for random same-class fit-set patients (not RF representatives), the proximity to their nearest same-class RF representative
rng_g = np.random.default_rng(CFG_GRADED["seed"])
null_prox = {}
for ci, c in enumerate(CLASS_NAMES):
    reps_c = rep_idx[rep_cls == c]
    pool = np.setdiff1d(np.flatnonzero(Y_train[:, ci] == 1), rep_idx)
    pick = rng_g.choice(pool, size=min(CFG_GRADED["n_null"], len(pool)), replace=False)
    null_prox[c] = (leaves[pick][:, None, :] == leaves[reps_c][None, :, :]).mean(axis=2).max(axis=1)


def graded_overlap(ppn, name):
    ppn = ppn[row_of_pid.reindex(ppn["patient_id"]).notna().to_numpy()].reset_index(drop=True)   # needs tabular features
    m = len(ppn)
    Xp = X_all_sel[row_of_pid[ppn["patient_id"]].to_numpy()].toarray().astype(np.float32)
    lp = proto_rf.apply(Xp)                                                  # (m, n_trees) leaves of the ProtoPregNet prototype patients
    cls = ppn["class"].to_numpy()
    prox = (lp[:, None, :] == leaves[rep_idx][None, :, :]).mean(axis=2)      # (m, r) proximity to every RF representative
    j = np.where(cls[:, None] == rep_cls[None, :], prox, -1.0).argmax(axis=1)   # nearest SAME-class RF representative
    best = prox[np.arange(m), j]
    # percentile within the null; ties get half credit so a random patient averages 0.5
    pct = np.array([(null_prox[c] < b).mean() + 0.5 * (null_prox[c] == b).mean() for c, b in zip(cls, best)])
    per = pd.DataFrame({"export": name, "ppn_proto_idx": ppn["ppn_proto_idx"], "class": cls, "ppn_patient_id": ppn["patient_id"],
                        "nearest_same_class_rf_proto": rep_proto[j], "prox_to_nearest": best, "null_percentile": pct})
    rows = []
    for c in CLASS_NAMES:
        s_ = per[per["class"] == c]
        n = len(s_)
        rows.append({"export": name, "class": c, "n_ppn": n, "mean_prox": s_["prox_to_nearest"].mean(),
                     "null_mean_prox": null_prox[c].mean(), "lift": s_["prox_to_nearest"].mean() / max(null_prox[c].mean(), 1e-9),
                     "mean_null_percentile": s_["null_percentile"].mean(),
                     "share_above_null_90th": float((s_["null_percentile"] >= 0.9).mean()) if n else np.nan,
                     "z_vs_random": float((s_["null_percentile"].mean() - 0.5) * np.sqrt(12 * n)) if n else np.nan})
    return per, pd.DataFrame(rows)


graded_per, graded_summary = [], []
for ex_name, path in CFG["ppn_exports"].items():
    for branch, ppn in load_ppn_prototypes(path).items():
        per, summ = graded_overlap(ppn, f"{ex_name}/{branch}")
        graded_per.append(per); graded_summary.append(summ)
graded_per, graded_summary = pd.concat(graded_per, ignore_index=True), pd.concat(graded_summary, ignore_index=True)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("mean_null_percentile: 0.5 = no closer to the RF prototypes than a random same-class patient; z_vs_random is a rough "
      "normal approximation (percentiles of random patients are ~uniform).")
print(graded_summary.round(3).to_string(index=False))
graded_per.round(3)

In [0]:
# ============================================================================
# Feature metadata for the design columns. Concept names come with the selected-features table (tabular_data_prep joined them), so no regex / concept join is needed.
# ============================================================================
_dom = sel["group"].str.split("_").str[0]                                            # cond | proc | drug | meas
_name = sel["concept_name"].where(sel["concept_name"].notna(), "concept " + sel["concept_id"].astype("Int64").astype(str))
_wk = sel["window_wk"].astype("Int64").astype(str)
FEATURE_META = pd.DataFrame({"feature": sel["design_col"], "role": sel["role"], "kind": _dom, "group": sel["group"], "stat": sel["stat"],
                             "concept_id": sel["concept_id"], "window_wk": sel["window_wk"], "concept_name": sel["concept_name"]})
FEATURE_META["display"] = np.select(
    [(sel["role"] == "presence") & (_dom == "meas"), sel["role"] == "presence", _dom == "meas"],
    [_name + " | measured in the window starting wk " + _wk, _name + " (" + _dom + ", present in the record)", _name + " | " + sel["stat"].astype(str) + ", window starting wk " + _wk],
    default=_name + " (" + _dom + ", first-occurrence week)")
print("design columns by role / kind:", FEATURE_META.groupby(["role", "kind"]).size().to_dict(),
      "| concept name available for", f"{FEATURE_META['concept_name'].notna().mean():.1%}", "of them")


In [0]:
# ============================================================================
# Named prototype inspection: profile with concept names, a one-line digest of every prototype, and a patient's features
# ============================================================================
_META_COLS = ["feature", "display", "kind", "concept_name", "window_wk", "stat"]
_proto_profile_raw = globals().get("_proto_profile_raw", proto_profile)        # keep the original so re-running this cell doesn't nest wrappers

def proto_profile(proto_idx, top=None):
    """As before, plus concept names: the features whose presence rate in the cluster differs most from its class."""
    out = _proto_profile_raw(proto_idx, top)
    return out.merge(FEATURE_META[_META_COLS], on="feature", how="left")[
        ["display"] + [c for c in out.columns if c != "feature"] + ["kind", "window_wk", "stat", "feature"]]


def proto_digest(top=5, min_group_rate=0.5):
    """One row per prototype: its most over-represented features (by name) among those present in >= min_group_rate of the cluster."""
    rows = []
    for _, r in proto_df.iterrows():
        prof = proto_profile(int(r["proto_idx"]), top=300)
        prof = prof[(prof["group_rate"] >= min_group_rate) & (prof["diff_vs_class"] > 0)].head(top)
        rows.append({"proto_idx": r["proto_idx"], "class": r["class"], "patient_id": r["patient_id"], "group_size": r["group_size"],
                     "top_features": " ; ".join(f"{d} [{g:.0%} vs {c:.0%}]" for d, g, c in zip(prof["display"], prof["group_rate"], prof[f"{r['class']}_rate"]))})
    return pd.DataFrame(rows)


def patient_features(train_idx, top=40):
    """Concepts / lab windows present in one fit-set patient (e.g. a prototype's representative), with names; value = first-occurrence week or the lab window's min."""
    j = np.flatnonzero(P_TRAIN[int(train_idx)])                                  # positions among the PRESENCE columns
    cols, vcol = PRES_IDX[j], PRES_VALUE_COL[j]
    val = np.where(vcol >= 0, X_train_sel[int(train_idx), np.maximum(vcol, 0)], np.nan)
    out = FEATURE_META.iloc[cols][["display", "kind", "window_wk", "stat"]].assign(value=val, feature=FEATURE_NAMES[cols])
    return out.sort_values(["kind", "value"]).head(top).reset_index(drop=True)


pd.set_option("display.max_colwidth", 200)
PROTO_TO_INSPECT = 17
proto_profile(PROTO_TO_INSPECT).round(3)

In [0]:
# ============================================================================
# STRUCTURE CHECK 1 -- are the RF prototypes of a class distinct, or copies of each other?
#   (a) pairwise RF proximity between a class's representatives vs random same-class sets of the same size
#   (b) overlap of their top enriched features (needs proto_profile from the inspection cell)
# Also defines prox_between / onehot_leaves, which the next two cells reuse.
# ============================================================================
CFG_STRUCT = {"seed": CFG["seed"], "block_rows": 5000, "n_random_sets": 300, "signature_top": 10, "top_pairs": 5}
rep_idx = proto_df["train_idx"].to_numpy()
rep_cls = proto_df["class"].to_numpy()
rep_proto = proto_df["proto_idx"].to_numpy()


def onehot_leaves(leaf_ids):
    """Sparse patient x leaf one-hot, same layout as L: proximity(a, b) = onehot(a) @ onehot(b).T / n_trees."""
    n = leaf_ids.shape[0]
    rows_ = np.repeat(np.arange(n), n_trees)
    cols_ = (leaf_ids + offsets[None, :]).ravel()
    return sp.csr_matrix((np.ones(rows_.size, dtype=np.float32), (rows_, cols_)), shape=(n, int(node_counts.sum())))


def prox_between(leaf_a, leaf_b):
    """(n_a, n_b) RF proximities (fraction of trees sharing a leaf), computed in row blocks of leaf_a."""
    Bt = onehot_leaves(leaf_b).T.tocsr()
    blk = CFG_STRUCT["block_rows"]
    return np.vstack([(onehot_leaves(leaf_a[s:s + blk]) @ Bt).toarray() / n_trees for s in range(0, len(leaf_a), blk)])


def mean_offdiag(P):
    n = len(P)
    return (P.sum() - np.trace(P)) / (n * (n - 1)) if n > 1 else np.nan


def signature(p, top):
    """Top features over-represented in prototype p's cluster relative to its class (by feature name)."""
    prof = proto_profile(int(p), top=300)
    return set(prof.loc[prof["diff_vs_class"] > 0, "feature"].head(top))


def redundancy_check(rep_idx, rep_cls, rep_proto, use_signatures=True):
    rng = np.random.default_rng(CFG_STRUCT["seed"])
    P = prox_between(leaves[rep_idx], leaves[rep_idx])
    np.fill_diagonal(P, 0.0)
    sig = [signature(p, CFG_STRUCT["signature_top"]) for p in rep_proto] if use_signatures else None
    jac = np.full((len(rep_idx), len(rep_idx)), np.nan)
    if use_signatures:
        for i in range(len(rep_idx)):
            for j in range(len(rep_idx)):
                u = len(sig[i] | sig[j])
                jac[i, j] = len(sig[i] & sig[j]) / u if u else np.nan
    rows, pairs = [], []
    for ci, c in enumerate(CLASS_NAMES):
        w = np.flatnonzero(rep_cls == c)
        if len(w) < 2:
            continue
        obs = mean_offdiag(P[np.ix_(w, w)])
        pool = np.setdiff1d(np.flatnonzero(Y_train[:, ci] == 1), rep_idx)
        null = np.array([mean_offdiag((leaves[s][:, None, :] == leaves[s][None, :, :]).mean(axis=2))
                         for s in (rng.choice(pool, len(w), replace=False) for _ in range(CFG_STRUCT["n_random_sets"]))])
        other = np.flatnonzero(rep_cls != c)
        rows.append({"class": c, "n_protos": len(w), "mean_pairwise_prox": obs, "random_set_mean_prox": null.mean(),
                     "frac_random_sets_less_similar": float((null < obs).mean()),
                     "mean_sig_jaccard_within": np.nanmean(jac[np.ix_(w, w)][~np.eye(len(w), dtype=bool)]) if use_signatures else np.nan,
                     "mean_sig_jaccard_vs_other_classes": np.nanmean(jac[np.ix_(w, other)]) if use_signatures else np.nan})
        iu = np.triu_indices(len(w), 1)
        order = np.argsort(-P[np.ix_(w, w)][iu])[:CFG_STRUCT["top_pairs"]]
        for k in order:
            a, b = w[iu[0][k]], w[iu[1][k]]
            pairs.append({"class": c, "proto_a": int(rep_proto[a]), "proto_b": int(rep_proto[b]), "prox": float(P[a, b]),
                          "sig_jaccard": float(jac[a, b]) if use_signatures else np.nan})
    class_mat = pd.DataFrame({b: [mean_offdiag(P[np.ix_(rep_cls == a, rep_cls == b)]) if a == b else P[np.ix_(rep_cls == a, rep_cls == b)].mean()
                                  for a in CLASS_NAMES] for b in CLASS_NAMES}, index=CLASS_NAMES)
    return pd.DataFrame(rows), pd.DataFrame(pairs), class_mat


redundancy_summary, redundancy_pairs, class_proximity = redundancy_check(rep_idx, rep_cls, rep_proto)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("frac_random_sets_less_similar: ~1 = prototypes are MORE alike than random same-class patients (redundant); ~0 = more spread out (distinct).")
print(redundancy_summary.round(3).to_string(index=False))
print("\nmean RF proximity between representatives, class x class (diagonal = within class):")
print(class_proximity.round(3).to_string())
print("\nmost similar prototype pairs per class (candidates for redundancy):")
print(redundancy_pairs.round(3).to_string(index=False))

In [0]:
# ============================================================================
# STRUCTURE CHECK 2 -- catchment: how much of each class do the prototypes cover, and does one prototype dominate?
#   Each class-c patient is assigned to its nearest class-c RF prototype (by proximity). Needs the previous cell's helpers.
# ============================================================================
CFG_CATCH = {"population": "val",        # "val" (honest, not used for fitting) or "train" (in-sample, inflated)
             "low_prox": 0.05,           # below this proximity a patient counts as NOT covered by any prototype of its class
             "n_random_sets": 100}

if CFG_CATCH["population"] == "val":
    leaves_pop, Y_pop = proto_rf.apply(X_val_sel), Y_val
else:
    leaves_pop, Y_pop = leaves, Y_train


def catchment_check(rep_idx, rep_cls, rep_proto):
    rng = np.random.default_rng(CFG_STRUCT["seed"])
    P_all = prox_between(leaves_pop, leaves[rep_idx])
    per_proto, per_class = [], []
    for ci, c in enumerate(CLASS_NAMES):
        w = np.flatnonzero(rep_cls == c)
        m = np.flatnonzero(Y_pop[:, ci] == 1)
        Pc = P_all[np.ix_(m, w)]
        best, nearest = Pc.max(axis=1), Pc.argmax(axis=1)
        covered = best >= CFG_CATCH["low_prox"]
        counts = np.bincount(nearest[covered], minlength=len(w))
        share = counts / max(counts.sum(), 1)
        for k, widx in enumerate(w):
            sel = covered & (nearest == k)
            per_proto.append({"proto_idx": int(rep_proto[widx]), "class": c, "n_assigned": int(counts[k]), "share_of_covered": float(share[k]),
                              "mean_prox_of_assigned": float(best[sel].mean()) if sel.any() else np.nan})
        # random exemplar sets of the same size drawn from the fit set: do the chosen prototypes cover more than random patients would?
        pool = np.flatnonzero(Y_train[:, ci] == 1)
        sets = np.stack([rng.choice(pool, len(w), replace=False) for _ in range(CFG_CATCH["n_random_sets"])])
        rand_best = prox_between(leaves_pop[m], leaves[sets.ravel()]).reshape(len(m), len(sets), len(w)).max(axis=2)   # (n_c, n_sets)
        per_class.append({"class": c, "n_patients": len(m), "n_protos": len(w), "uncovered_share": float(1 - covered.mean()),
                          "top1_share": float(share.max()), "effective_n_protos": float(1 / (share ** 2).sum()) if counts.sum() else np.nan,
                          "max_prox_p10": np.percentile(best, 10), "max_prox_median": np.median(best), "max_prox_p90": np.percentile(best, 90),
                          "mean_max_prox": float(best.mean()), "random_sets_mean_max_prox": float(rand_best.mean()),
                          "frac_random_sets_covering_less": float((rand_best.mean(axis=0) < best.mean()).mean())})
    return pd.DataFrame(per_proto), pd.DataFrame(per_class)


catch_proto, catch_class = catchment_check(rep_idx, rep_cls, rep_proto)
print(f"population: {CFG_CATCH['population']} | covered = proximity >= {CFG_CATCH['low_prox']} to some prototype of the patient's class")
print("effective_n_protos = 1 / sum(share^2): equals n_protos when all prototypes claim equal shares, ~1 when one dominates.")
print(catch_class.round(3).to_string(index=False))
catch_proto.round(3)

In [0]:
# ============================================================================
# STRUCTURE CHECK 4 -- prototype-only predictor on VALIDATION: how much of the forest's signal do the prototypes carry?
#   score_c(x) = max proximity of x to class-c prototype representatives (and, as a variant, the mean over them).
#   Compared with the forest's own AUROC and with random same-class exemplar sets of the same size. Test is not touched.
# ============================================================================
CFG_PRED = {"n_random_sets": 100}

def predictor_check(rep_idx, rep_cls, X_pop, Y_pop):
    rng = np.random.default_rng(CFG_STRUCT["seed"])
    lv = proto_rf.apply(X_pop)
    P_all = prox_between(lv, leaves[rep_idx])
    forest = predict_matrix(proto_rf, X_pop)
    rows, rand_by_class = [], {}
    for ci, c in enumerate(CLASS_NAMES):
        w = np.flatnonzero(rep_cls == c)
        y = Y_pop[:, ci]
        pool = np.flatnonzero(Y_train[:, ci] == 1)
        sets = np.stack([rng.choice(pool, len(w), replace=False) for _ in range(CFG_PRED["n_random_sets"])])
        Pr = prox_between(lv, leaves[sets.ravel()]).reshape(len(y), len(sets), len(w))
        rand = np.array([roc_auc_score(y, Pr[:, s, :].max(axis=1)) for s in range(len(sets))])
        rand_by_class[c] = rand
        rows.append({"class": c, "n_protos": len(w), "auroc_prototypes_max": roc_auc_score(y, P_all[:, w].max(axis=1)),
                     "auroc_prototypes_mean": roc_auc_score(y, P_all[:, w].mean(axis=1)), "auroc_forest": roc_auc_score(y, forest[:, ci]),
                     "random_sets_auroc_mean": rand.mean(), "random_sets_p5": np.percentile(rand, 5), "random_sets_p95": np.percentile(rand, 95)})
    out = pd.DataFrame(rows)
    macro_rand = np.mean([rand_by_class[c] for c in CLASS_NAMES], axis=0)
    macro = {"class": "macro", "n_protos": int(out["n_protos"].sum())}
    for k in ["auroc_prototypes_max", "auroc_prototypes_mean", "auroc_forest", "random_sets_auroc_mean", "random_sets_p5", "random_sets_p95"]:
        macro[k] = float(out[k].mean())
    macro["random_sets_auroc_mean"], macro["random_sets_p5"], macro["random_sets_p95"] = macro_rand.mean(), np.percentile(macro_rand, 5), np.percentile(macro_rand, 95)
    out = pd.concat([out, pd.DataFrame([macro])], ignore_index=True)
    out["frac_random_sets_below_prototypes"] = [float((rand_by_class[c] < r).mean()) if c in rand_by_class else float((macro_rand < macro["auroc_prototypes_max"]).mean())
                                                for c, r in zip(out["class"], out["auroc_prototypes_max"])]
    return out

predictor_summary = predictor_check(rep_idx, rep_cls, X_val_sel, Y_val)
print("AUROC on validation. frac_random_sets_below_prototypes: share of random exemplar sets the chosen prototypes beat (0.5 = no better than random).")
print(predictor_summary.round(3).to_string(index=False))

In [0]:
# ============================================================================
# PROTOTYPE-ONLY TEST on the held-out TEST split: how much of the forest's class structure do the RF prototypes carry?
#   score_c(x) = the highest RF proximity between test patient x and the class-c prototype patients (0 if x shares no leaf with any of them).
#   Its AUROC for label c is compared with
#     - random exemplars : same-size sets of random fit-set patients of class c, scored the same way (the null for "no better than chance exemplars")
#     - kNN, all train   : proximity-weighted vote of the k most proximal fit-set patients, same proximity space (the STRUCTURE ceiling: what the
#                          trees' neighbourhoods hold before they are compressed to a few exemplars)
#     - the forest       : its own predicted probabilities (the model ceiling)
#   Patients with score 0 are tied, so the AUROC is also limited by how many patients the prototypes reach at all: the coverage columns and
#   auroc_cap_at_coverage (the best AUROC possible at that coverage even if every reached patient were ranked perfectly) show how much.
#   Run it ONCE for a number you cite: the test split was not used to fit the forest, choose its hyperparameters, or extract the prototypes.
#   Needs from earlier cells: proto_rf, leaves, n_trees, offsets, node_counts, proto_df, X_test_sel, Y_test, Y_train, predict_matrix, CLASS_NAMES, CFG.
# ============================================================================
from scipy.stats import rankdata

CFG_PTEST = {
    "n_random_sets": 1000,        # random exemplar sets per class (the null distribution)
    "n_random_sets_boot": 100,    # of those, how many enter each bootstrap resample (for the paired difference to the null; cost grows with it)
    "n_bootstrap": 1000, "ci": 0.95,
    "knn_k": 20,                  # neighbours in the structure ceiling (same k as the cluster size used for the prototypes)
    "block_rows": 1000,           # memory knob only
    "seed": CFG["seed"],
}


def _onehot(leaf_ids):
    """Sparse patient x leaf one-hot (same layout as L): proximity(a, b) = onehot(a) @ onehot(b).T / n_trees."""
    n = leaf_ids.shape[0]
    return sp.csr_matrix((np.ones(n * n_trees, dtype=np.float32), (np.repeat(np.arange(n), n_trees), (leaf_ids + offsets[None, :]).ravel())),
                         shape=(n, int(node_counts.sum())))


def _prox(leaf_a, leaf_b, block=CFG_PTEST["block_rows"]):
    """(n_a, n_b) fraction of trees in which a patient of A and a patient of B share a leaf."""
    Bt = _onehot(leaf_b).T.tocsr()
    return np.vstack([(_onehot(leaf_a[s:s + block]) @ Bt).toarray() / n_trees for s in range(0, len(leaf_a), block)])


def auroc_cols(S, y):
    """AUROC of every column of S (n, m) against the binary label y; ties get half credit (identical to sklearn's roc_auc_score per column)."""
    n_pos = int(y.sum()); n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.full(S.shape[1], np.nan)
    R = rankdata(S, axis=0)
    return (R[y == 1].sum(axis=0) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


C = len(CLASS_NAMES)
n_test = len(Y_test)
lt = proto_rf.apply(X_test_sel)                                                  # (n_test, n_trees): leaves of the held-out patients
rep_idx, rep_cls = proto_df["train_idx"].to_numpy(), proto_df["class"].to_numpy()
rng = np.random.default_rng(CFG_PTEST["seed"])

# ---- scores ----
P_rep = _prox(lt, leaves[rep_idx])                                               # (n_test, n_prototypes)
S_proto = np.column_stack([P_rep[:, rep_cls == c].max(axis=1) for c in CLASS_NAMES])
S_forest = np.round(predict_matrix(proto_rf, X_test_sel), 9)                    # predict_proba with n_jobs > 1 is not bitwise deterministic (float summation order); rounding makes the ranking reproducible
k, blk = CFG_PTEST["knn_k"], CFG_PTEST["block_rows"]
S_knn = np.zeros((n_test, C), dtype=np.float32)
for s in range(0, n_test, blk):                                                  # kNN over ALL fit-set patients, in row blocks
    P = _prox(lt[s:s + blk], leaves)
    top = np.argpartition(-P, k, axis=1)[:, :k]
    w = np.take_along_axis(P, top, axis=1)                                       # proximity weights of the k nearest
    S_knn[s:s + blk] = (w[:, :, None] * Y_train[top]).sum(axis=1) / np.maximum(w.sum(axis=1), 1e-12)[:, None]    # weighted class fraction (0 if no shared leaf)
S_null = {}                                                                      # class -> (n_test, n_random_sets) scores of random exemplar sets
for ci, c in enumerate(CLASS_NAMES):
    r_c = int((rep_cls == c).sum())
    pool = np.setdiff1d(np.flatnonzero(Y_train[:, ci] == 1), rep_idx)             # fit-set patients of class c that are not prototypes
    sets = np.stack([rng.choice(pool, r_c, replace=False) for _ in range(CFG_PTEST["n_random_sets"])])
    S_null[c] = _prox(lt, leaves[sets.ravel()]).reshape(n_test, CFG_PTEST["n_random_sets"], r_c).max(axis=2)

# ---- point estimates on the full test set ----
auc = {"proto": np.array([auroc_cols(S_proto[:, [i]], Y_test[:, i])[0] for i in range(C)]),
       "knn": np.array([auroc_cols(S_knn[:, [i]], Y_test[:, i])[0] for i in range(C)]),
       "forest": np.array([auroc_cols(S_forest[:, [i]], Y_test[:, i])[0] for i in range(C)])}
null_auc = np.stack([auroc_cols(S_null[c], Y_test[:, i]) for i, c in enumerate(CLASS_NAMES)], axis=1)      # (n_random_sets, C)

# ---- paired bootstrap over TEST patients (same resample for every method) ----
rng_b = np.random.default_rng(CFG_PTEST["seed"] + 1)
B, nb = CFG_PTEST["n_bootstrap"], CFG_PTEST["n_random_sets_boot"]
boot = {key: np.full((B, C), np.nan) for key in ("proto", "knn", "forest", "null")}
for b in range(B):
    idx = rng_b.integers(0, n_test, n_test)
    for ci, c in enumerate(CLASS_NAMES):
        a = auroc_cols(np.column_stack([S_proto[idx, ci], S_knn[idx, ci], S_forest[idx, ci], S_null[c][idx, :nb]]), Y_test[idx, ci])
        boot["proto"][b, ci], boot["knn"][b, ci], boot["forest"][b, ci], boot["null"][b, ci] = a[0], a[1], a[2], a[3:].mean()
lo_q, hi_q = (1 - CFG_PTEST["ci"]) / 2 * 100, (1 + CFG_PTEST["ci"]) / 2 * 100

# ---- coverage: how many test patients the prototypes reach at all ----
nz = S_proto > 0
cov_pos = np.array([nz[Y_test[:, i] == 1, i].mean() for i in range(C)]); cov_neg = np.array([nz[Y_test[:, i] == 0, i].mean() for i in range(C)])
cap = cov_pos + 0.5 * (1 - cov_pos) * (1 - cov_neg)                              # best AUROC at this coverage: reached positives above reached negatives above the tied rest
zero_share = (~nz).mean(axis=0)

# ---- result table: one row per class, then the two macro averages ----
groups = {**{c: [i] for i, c in enumerate(CLASS_NAMES)}, "macro (all labels)": list(range(C)), "macro (outcomes only)": [CLASS_NAMES.index(c) for c in OUTCOME_NAMES]}
rows = []
for name, cols in groups.items():
    pt = {m: float(np.nanmean(auc[m][cols])) for m in auc}
    bt = {m: np.nanmean(boot[m][:, cols], axis=1) for m in boot}
    nul = np.nanmean(null_auc[:, cols], axis=1)                                  # per-set (macro) AUROC of the random exemplars
    delta = bt["proto"] - bt["null"]
    ci_ = lambda v: tuple(np.nanpercentile(v, [lo_q, hi_q]))
    rows.append({"label": name, "prototypes": pt["proto"], "ci_lo": ci_(bt["proto"])[0], "ci_hi": ci_(bt["proto"])[1],
                 "random_mean": float(nul.mean()), "random_p5": float(np.percentile(nul, 5)), "random_p95": float(np.percentile(nul, 95)),
                 "share_random_beaten": float((nul < pt["proto"]).mean() + 0.5 * (nul == pt["proto"]).mean()),
                 "delta_vs_random": float(np.nanmean(delta)), "delta_ci_lo": ci_(delta)[0], "delta_ci_hi": ci_(delta)[1],
                 "knn_all_train": pt["knn"], "knn_ci_lo": ci_(bt["knn"])[0], "knn_ci_hi": ci_(bt["knn"])[1],
                 "forest": pt["forest"], "forest_ci_lo": ci_(bt["forest"])[0], "forest_ci_hi": ci_(bt["forest"])[1],
                 "zero_score_share": float(zero_share[cols].mean()), "reached_pos": float(cov_pos[cols].mean()), "reached_neg": float(cov_neg[cols].mean()),
                 "auroc_cap_at_coverage": float(cap[cols].mean())})
proto_test_summary = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(f"TEST split, n = {n_test:,} | {len(proto_df)} prototypes | {CFG_PTEST['n_random_sets']} random exemplar sets per class | {B} paired bootstrap resamples ({CFG_PTEST['ci']:.0%} CI)")
print("prototypes = prototype-only AUROC | random_* = same-size random exemplars | delta_vs_random = prototypes - random mean (paired CI) | knn_all_train = structure ceiling | forest = model ceiling")
print("zero_score_share = test patients with no shared leaf with any prototype of the class | reached_pos / reached_neg = share of positives / negatives with a nonzero score\n")
print(proto_test_summary.round(3).to_string(index=False))
proto_test_summary

In [0]:
# ============================================================================
# BLOOD PRESSURE in the prototypes of one class: how many have systolic AND diastolic results, and how early
#   Unit = the PROTOTYPE PATIENT (the real patient standing in for a cluster). The cluster's share is reported alongside as a robustness check.
#   Source = the FULL stored matrix (every lab window), not just the selected design columns, so a BP window that feature selection dropped still counts.
#   Lab features are WINDOW summaries (tmeta['window_len_days'] long, one every stride), so "before week 20" is only certain if the whole window ends by
#   week 20. Two rules are reported: STRICT = a window with both results lying entirely in weeks 0-20; OVERLAP = a window with both results that overlaps
#   weeks 0-20 (can include results a few weeks after week 20, or just before pregnancy).
#   Baselines (all patients of the class, all fit-set patients) are reported too: if nearly everyone has BP results, "8 of 9 have BP" says little.
#   Needs from earlier cells: X_full, feature_names, fit_mask, proto_df, PROTO_GROUPS, tmeta, spark.
# ============================================================================
import re
from pyspark.sql import functions as F

CFG_BP = {
    "class": "pe_ghtn",                                   # whose prototypes
    "concept_table": "catalog.omop.concept",         # (concept_id, concept_name): finds the BP concepts by name
    "systolic_regex": r"systolic", "diastolic_regex": r"diastolic", "name_must_match": r"blood pressure", "exclude_regex": r"pulmonary|fetal",
    "concept_ids": None,                                  # optional explicit override {"systolic": [ids], "diastolic": [ids]}: skips the name search
    "week_cutoff": 20,                                    # 'early' = before this gestational week
    "abnormal_sbp": 140, "abnormal_dbp": 90,              # a window max at or above either threshold = an abnormal-range result
}

# ---- lab columns of the stored matrix: concept, window start week, statistic ----
lab_re = re.compile(r"^meas_(-?\d+)_wk(-?\d+)_(min|max)$")
parsed = [(i, *(int(m.group(1)), int(m.group(2)), m.group(3))) for i, nm in enumerate(feature_names) if (m := lab_re.match(nm))]
col_idx = np.array([p[0] for p in parsed]); col_concept = np.array([p[1] for p in parsed]); col_wk = np.array([p[2] for p in parsed]); col_stat = np.array([p[3] for p in parsed])

# ---- which concepts are systolic / diastolic blood pressure (printed so you can check them) ----
if CFG_BP["concept_ids"]:
    bp_ids = {k: set(map(int, v)) for k, v in CFG_BP["concept_ids"].items()}
else:
    ids = pd.DataFrame({"concept_id": np.unique(col_concept).astype("int64")})
    names = (spark.table(CFG_BP["concept_table"]).select(F.col("concept_id").cast("long").alias("concept_id"), "concept_name")
             .join(F.broadcast(spark.createDataFrame(ids)), "concept_id").toPandas())
    nm = names["concept_name"].astype(str)
    base = nm.str.contains(CFG_BP["name_must_match"], case=False, regex=True) & ~nm.str.contains(CFG_BP["exclude_regex"], case=False, regex=True)
    bp_ids = {"systolic": set(names.loc[base & nm.str.contains(CFG_BP["systolic_regex"], case=False, regex=True), "concept_id"].astype(int)),
              "diastolic": set(names.loc[base & nm.str.contains(CFG_BP["diastolic_regex"], case=False, regex=True), "concept_id"].astype(int))}
    for k in bp_ids:
        print(f"{k} concepts counted:", names[names["concept_id"].isin(bp_ids[k])][["concept_id", "concept_name"]].values.tolist())
    for k in bp_ids:
        if not bp_ids[k]:
            raise ValueError(f"no {k} blood-pressure concept found among the stored lab concepts; candidates containing 'pressure': "
                             f"{names[nm.str.contains('pressure', case=False)]['concept_name'].head(15).tolist()} -> adjust the regexes or set CFG_BP['concept_ids']")

# ---- per fit-set patient: which windows have systolic / diastolic results ----
fit_rows = np.flatnonzero(fit_mask)
Xr = X_full[fit_rows]                                                            # fit-set rows (same order as proto_df['train_idx'])
wks = np.unique(col_wk[np.isin(col_concept, list(bp_ids["systolic"] | bp_ids["diastolic"]))])
def window_presence(concepts):
    cols = {w: col_idx[np.isin(col_concept, list(concepts)) & (col_wk == w)] for w in wks}
    return np.column_stack([(Xr[:, c].getnnz(axis=1) > 0) if len(c) else np.zeros(Xr.shape[0], bool) for c in cols.values()])
SYS, DIA = window_presence(bp_ids["systolic"]), window_presence(bp_ids["diastolic"])         # (n_fit, n_windows) bool
BOTH = SYS & DIA
wlen = tmeta["window_len_days"] / tmeta["step_days"]                             # window length in weeks
cut = CFG_BP["week_cutoff"]
strict_w = (wks >= 0) & (wks + wlen <= cut)                                       # window entirely inside weeks 0..cutoff
overlap_w = (wks < cut) & (wks + wlen > 0)                                        # window overlapping weeks 0..cutoff
flags = {"both_any": BOTH.any(axis=1), "early_strict": BOTH[:, strict_w].any(axis=1), "early_overlap": BOTH[:, overlap_w].any(axis=1)}

def window_max(concepts):
    """Per fit-set patient: the largest window maximum of any of the concepts (NaN if none)."""
    cols = col_idx[np.isin(col_concept, list(concepts)) & (col_stat == "max")]
    sub = Xr[:, cols].tocsr() if len(cols) else None
    if sub is None: return np.full(Xr.shape[0], np.nan)
    mx = np.full(Xr.shape[0], -np.inf); np.maximum.at(mx, np.repeat(np.arange(sub.shape[0]), np.diff(sub.indptr)), sub.data)
    return np.where(np.isinf(mx), np.nan, mx)
SBP, DBP = window_max(bp_ids["systolic"]), window_max(bp_ids["diastolic"])
flags["abnormal"] = (SBP >= CFG_BP["abnormal_sbp"]) | (DBP >= CFG_BP["abnormal_dbp"])

# ---- per-prototype table ----
cls_idx = CLASS_NAMES.index(CFG_BP["class"])
protos = proto_df[proto_df["class"] == CFG_BP["class"]]
rows = []
for _, r in protos.iterrows():
    i = int(r["train_idx"]); members = PROTO_GROUPS[int(r["proto_idx"])]
    both_w = wks[BOTH[i]]
    rows.append({"proto_idx": int(r["proto_idx"]), "patient_id": r["patient_id"], "has_systolic": bool(SYS[i].any()), "has_diastolic": bool(DIA[i].any()),
                 "has_both": bool(flags["both_any"][i]), "windows_with_both": both_w.tolist(), "earliest_window_start_wk": int(both_w.min()) if len(both_w) else None,
                 "early_strict": bool(flags["early_strict"][i]), "early_overlap": bool(flags["early_overlap"][i]),
                 "max_sbp": SBP[i], "max_dbp": DBP[i], "abnormal_range": bool(flags["abnormal"][i]),
                 "cluster_share_both": float(flags["both_any"][members].mean()), "cluster_share_early_strict": float(flags["early_strict"][members].mean())})
bp_prototypes = pd.DataFrame(rows)

# ---- summary with baselines ----
in_class = Y_train[:, cls_idx] == 1
n = len(bp_prototypes)
summary = pd.DataFrame({
    "prototypes (n)": [int(bp_prototypes[c].sum()) for c in ("has_both", "early_strict", "early_overlap", "abnormal_range")],
    "of": n,
    f"all {CFG_BP['class']} patients": [float(flags[k][in_class].mean()) for k in ("both_any", "early_strict", "early_overlap", "abnormal")],
    "all fit-set patients": [float(flags[k].mean()) for k in ("both_any", "early_strict", "early_overlap", "abnormal")],
}, index=["systolic AND diastolic results (any window)", f"... in a window entirely before week {cut} (strict)",
          f"... in a window overlapping weeks 0-{cut} (overlap)", f"abnormal range (max window SBP >= {CFG_BP['abnormal_sbp']} or DBP >= {CFG_BP['abnormal_dbp']})"])
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 120)
print(f"{n} {CFG_BP['class']} prototypes | windows are {wlen:g} weeks long, starting every {np.diff(wks).min() if len(wks) > 1 else '?'} weeks; result = a stored window summary, not a single reading\n")
print(summary.round(3).to_string())
print("\nper prototype:"); print(bp_prototypes.round(1).to_string(index=False))

In [0]:
# ============================================================================
# LAB PANEL in the prototypes of one class (platelets, creatinine): result available? abnormal? -- vs the class and vs the whole fit set
#   Same logic as the blood-pressure cell: unit = the PROTOTYPE PATIENT; source = the FULL stored matrix (every lab window), not only the selected
#   columns; the cluster's share is reported as a robustness check. Concepts are given by id and printed with their names so you can check them.
#   DIRECTION matters: a LOW platelet count is the abnormal one, so it is read from the window MINIMUM; a HIGH creatinine is read from the window MAXIMUM.
#   Defaults are the ACOG "preeclampsia with severe features" cut-offs (platelets < 100 x10^3/uL, creatinine > 1.1 mg/dL): confirm them, and confirm the
#   UNITS of the stored values (the printout shows each lab's value range and warns if the median is outside the range you expect).
#   These are WINDOW summaries from the whole stored record, including the diagnostic work-up: they describe the prototypes, they do not show that an
#   abnormal value preceded diagnosis. Labs are also ordered more when preeclampsia is suspected, so availability is itself informative.
#   Needs from earlier cells: X_full, feature_names, fit_mask, proto_df, PROTO_GROUPS, Y_train, CLASS_NAMES, spark.
# ============================================================================
import re
from pyspark.sql import functions as F
from scipy.stats import beta, binomtest

CFG_LAB = {
    "class": "pe_ghtn",                                   # whose prototypes
    "concept_table": "catalog.omop.concept",         # only used to print the names of the concept ids below
    "labs": {                                             # one entry per lab
        "platelets":  {"concept_ids": [3024929], "abnormal": "low",  "threshold": 100, "unit_check_median": (100, 600)},   # abnormal = window min < threshold
        "creatinine": {"concept_ids": [3016723], "abnormal": "high", "threshold": 1.1, "unit_check_median": (0.3, 2.0)},    # abnormal = window max > threshold
    },
}

# ---- lab columns of the stored matrix ----
lab_re = re.compile(r"^meas_(-?\d+)_wk(-?\d+)_(min|max)$")
parsed = [(i, int(m.group(1)), m.group(3)) for i, nm in enumerate(feature_names) if (m := lab_re.match(nm))]
col_idx = np.array([p[0] for p in parsed]); col_concept = np.array([p[1] for p in parsed]); col_stat = np.array([p[2] for p in parsed])

ids_all = sorted({c for v in CFG_LAB["labs"].values() for c in v["concept_ids"]})
named = spark.table(CFG_LAB["concept_table"]).filter(F.col("concept_id").isin(ids_all)).select("concept_id", "concept_name").toPandas()
print("concepts counted:", named.values.tolist(), "\n")

fit_rows = np.flatnonzero(fit_mask)
Xr = X_full[fit_rows]                                                            # fit-set rows (same order as proto_df['train_idx'])

def lab_patient_summary(concept_ids, direction):
    """Per fit-set patient: (any result in any window, the worst window value in the abnormal direction, all stored window values of the lab)."""
    in_lab = np.isin(col_concept, list(concept_ids))
    if not in_lab.any():
        raise ValueError(f"none of the concept ids {sorted(concept_ids)} is in the stored matrix as meas_<id>_wk<w>_min/max: check the ids")
    sub_all = Xr[:, col_idx[in_lab]].tocsr()
    has = np.diff(sub_all.indptr) > 0                                              # presence = a stored entry (structure), never value != 0
    stat = "min" if direction == "low" else "max"
    sub = Xr[:, col_idx[in_lab & (col_stat == stat)]].tocsr()
    worst = np.full(Xr.shape[0], np.inf if direction == "low" else -np.inf)
    (np.minimum if direction == "low" else np.maximum).at(worst, np.repeat(np.arange(sub.shape[0]), np.diff(sub.indptr)), sub.data)
    return has, np.where(np.isinf(worst), np.nan, worst), sub_all.data

cls_idx = CLASS_NAMES.index(CFG_LAB["class"])
in_class = Y_train[:, cls_idx] == 1
protos = proto_df[proto_df["class"] == CFG_LAB["class"]].reset_index(drop=True)
n = len(protos)
LAB_PATIENT, summary_rows = {}, []
proto_cols = {"proto_idx": protos["proto_idx"].astype(int).to_numpy(), "patient_id": protos["patient_id"].to_numpy()}
cp = lambda k: (beta.ppf(0.025, k, n - k + 1) if k > 0 else 0.0, beta.ppf(0.975, k + 1, n - k) if k < n else 1.0)       # Clopper-Pearson 95% CI

for lab, cfg in CFG_LAB["labs"].items():
    has, worst, vals = lab_patient_summary(cfg["concept_ids"], cfg["abnormal"])
    abnormal = (worst < cfg["threshold"]) if cfg["abnormal"] == "low" else (worst > cfg["threshold"])        # NaN compares False: no result = not abnormal
    LAB_PATIENT[lab] = {"has": has, "worst": worst, "abnormal": abnormal}
    lo, hi = np.percentile(vals, [1, 99]); med = float(np.median(vals)); ok_lo, ok_hi = cfg["unit_check_median"]
    print(f"{lab}: stored window values p1 {lo:g} | median {med:g} | p99 {hi:g}" + ("" if ok_lo <= med <= ok_hi else f"   <-- WARNING: median outside the expected {ok_lo:g}-{ok_hi:g}: check the units / the concept before trusting the threshold"))
    idx = protos["train_idx"].astype(int).to_numpy(); members = [PROTO_GROUPS[int(p)] for p in protos["proto_idx"]]
    proto_cols.update({f"{lab}_has": has[idx], f"{lab}_worst": worst[idx], f"{lab}_abnormal": abnormal[idx],
                       f"{lab}_cluster_share_has": [float(has[m].mean()) for m in members], f"{lab}_cluster_share_abnormal": [float(abnormal[m].mean()) for m in members]})
    sign = "<" if cfg["abnormal"] == "low" else ">"
    for what, arr in ((f"{lab}: result in any window", has), (f"{lab}: abnormal ({'min' if cfg['abnormal'] == 'low' else 'max'} {sign} {cfg['threshold']:g})", abnormal)):
        k = int(arr[idx].sum()); rate = float(arr[in_class].mean()); ci = cp(k)
        summary_rows.append({"measure": what, "prototypes (n)": k, "of": n, "ci_lo": ci[0], "ci_hi": ci[1], f"all {CFG_LAB['class']} patients": rate,
                             "expected_n": n * rate, "p_binomial": binomtest(k, n, rate).pvalue, "all fit-set patients": float(arr.mean())})
lab_prototypes = pd.DataFrame(proto_cols)
lab_summary = pd.DataFrame(summary_rows).set_index("measure")

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print(f"\n{n} {CFG_LAB['class']} prototypes | prototypes (n) with a 95% Clopper-Pearson CI; p_binomial = exact two-sided test against the class rate (n is small: read it as a consistency check)\n")
print(lab_summary.round(3).to_string())
print("\nper prototype:"); print(lab_prototypes.round(2).to_string(index=False))

In [0]:
# ============================================================================
# AGGREGATE FEATURE REPRESENTATION in the prototypes of one class, vs prevalence
#   Feature space = the forest's own: the SELECTED presence columns, rolled up to CONCEPT level (a concept is present if its event is present, or any of its
#   selected lab windows is). For each concept: how often is it present in the prototypes' clusters (all clusters of the class pooled) and in the representative
#   patients, compared with how common it is in the class, in controls and in all fit-set patients?
#   Test: exact binomial test of the pooled cluster count against the class rate. Its null is "the clusters are random subsets of the class", so it asks
#   whether the clusters are more (or less) distinctive than chance. It does NOT say a feature is important to the forest, and distinctive-within-class is not
#   outcome-related by itself: class_vs_control_lift (class rate / control rate) is the outcome link. Benjamini-Hochberg across the concepts tested.
#   Needs from earlier cells: P_TRAIN, PRES_IDX, FEATURE_META, proto_df, PROTO_GROUPS, Y_train, CLASS_NAMES.
# ============================================================================
from scipy.stats import binomtest

CFG_AGG = {
    "class": "pe_ghtn", "proto_idx_range": (17, 25),      # inclusive; the prototypes of that class to aggregate
    "min_class_rate": 0.02,       # concepts present in < 2% of the class are not tested (their lift is noise)
    "outcome_lift_min": 1.5,      # 'outcome-linked' = class rate / control rate at least this
    "alpha": 0.05,                # Benjamini-Hochberg threshold
    "top": 15,                    # rows shown per table
}
assert CLASS_NAMES[0] == "control"
protos = proto_df[(proto_df["class"] == CFG_AGG["class"]) & proto_df["proto_idx"].between(*CFG_AGG["proto_idx_range"])]
if protos.empty:
    raise ValueError(f"no '{CFG_AGG['class']}' prototypes with proto_idx in {CFG_AGG['proto_idx_range']}: have {proto_df.groupby('class')['proto_idx'].agg(['min', 'max']).to_dict('index')}")
in_class, in_ctrl = Y_train[:, CLASS_NAMES.index(CFG_AGG["class"])] == 1, Y_train[:, 0] == 1

# ---- roll the selected presence columns up to concepts ----
meta = FEATURE_META.iloc[PRES_IDX].reset_index(drop=True)                         # one row per presence column, same order as P_TRAIN's columns
meta["concept_key"] = meta["kind"].astype(str) + "_" + meta["concept_id"].astype("Int64").astype(str)
codes, keys = pd.factorize(meta["concept_key"])
M = sp.csr_matrix((np.ones(len(codes), dtype=np.float32), (np.arange(len(codes)), codes)), shape=(len(codes), len(keys)))
C = (sp.csr_matrix(P_TRAIN.astype(np.float32)) @ M).toarray() > 0                 # (n_fit, n_concepts): present in any of the concept's selected columns
info = meta.groupby("concept_key", sort=False).agg(kind=("kind", "first"), concept_name=("concept_name", "first"), n_cols=("feature", "size")).loc[keys]

# ---- rates: clusters pooled, representatives, class, controls, everyone ----
members = [PROTO_GROUPS[int(p)] for p in protos["proto_idx"]]
pool = np.unique(np.concatenate(members))
if len(pool) != sum(len(m) for m in members):
    print(f"NOTE: the clusters overlap ({sum(len(m) for m in members)} slots, {len(pool)} unique patients): the pool uses the unique patients")
reps = protos["train_idx"].astype(int).to_numpy()
rate_cluster, k_cluster = C[pool].mean(axis=0), C[pool].sum(axis=0)
class_rate, ctrl_rate = C[in_class].mean(axis=0), C[in_ctrl].mean(axis=0)
per_cluster = np.stack([C[m].mean(axis=0) for m in members])                      # (n_prototypes, n_concepts)
tab = info.assign(reps_n=C[reps].sum(axis=0), cluster_rate=rate_cluster, class_rate=class_rate, control_rate=ctrl_rate, overall_rate=C.mean(axis=0),
                  diff_vs_class=rate_cluster - class_rate, lift_vs_class=rate_cluster / np.maximum(class_rate, 1e-12),
                  n_clusters_above=(per_cluster > class_rate).sum(axis=0),
                  class_vs_control_lift=class_rate / np.maximum(ctrl_rate, 1.0 / in_ctrl.sum()))
tab["display"] = np.where(tab["concept_name"].notna(), tab["concept_name"].astype(str) + " (" + tab["kind"].astype(str) + ")", tab.index.astype(str))

# ---- exact binomial test of the pooled cluster count against the class rate, BH-adjusted over the concepts tested ----
tested = (tab["class_rate"] >= CFG_AGG["min_class_rate"]).to_numpy()
p = np.array([binomtest(int(k), len(pool), r).pvalue for k, r in zip(k_cluster[tested], class_rate[tested])])
order = np.argsort(p); ranked = p[order] * len(p) / (np.arange(len(p)) + 1)
q = np.empty_like(p); q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
tab["p_binomial"], tab["q_bh"] = np.nan, np.nan
tab.loc[tested, "p_binomial"], tab.loc[tested, "q_bh"] = p, q
agg_features = tab[tested].sort_values("diff_vs_class", ascending=False)           # every concept tested, most over-represented first

# ---- report ----
n_p = len(protos); sig = agg_features["q_bh"] < CFG_AGG["alpha"]
cols = ["display", "reps_n", "cluster_rate", "class_rate", "control_rate", "overall_rate", "diff_vs_class", "lift_vs_class", "n_clusters_above", "class_vs_control_lift", "p_binomial", "q_bh"]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 70)
show = lambda df: df[cols].head(CFG_AGG["top"]).rename(columns={"reps_n": f"reps_n (of {n_p})", "n_clusters_above": f"clusters_above (of {n_p})"}).round(3).to_string(index=False)
print(f"{n_p} '{CFG_AGG['class']}' prototypes (proto_idx {CFG_AGG['proto_idx_range'][0]}-{CFG_AGG['proto_idx_range'][1]}) | clusters pooled: {len(pool)} patients | class n = {int(in_class.sum()):,}, control n = {int(in_ctrl.sum()):,}, all = {len(C):,}")
print(f"{len(agg_features)} concepts tested (class rate >= {CFG_AGG['min_class_rate']:.0%}); q < {CFG_AGG['alpha']}: {int((sig & (agg_features['diff_vs_class'] > 0)).sum())} over-represented, "
      f"{int((sig & (agg_features['diff_vs_class'] < 0)).sum())} under-represented | share of tested concepts with cluster rate above class rate: {(agg_features['diff_vs_class'] > 0).mean():.0%} (about half is expected for random clusters)\n")
print(f"MOST OVER-REPRESENTED in the clusters vs the class (by rate difference):\n{show(agg_features[agg_features['diff_vs_class'] > 0])}\n")
print(f"... of those over-represented, also OUTCOME-LINKED (class rate / control rate >= {CFG_AGG['outcome_lift_min']}):\n{show(agg_features[(agg_features['diff_vs_class'] > 0) & (agg_features['class_vs_control_lift'] >= CFG_AGG['outcome_lift_min'])])}\n")
print(f"MOST UNDER-REPRESENTED (what the clusters lack relative to the class; check this before reading the top table as clinical content):\n{show(agg_features[agg_features['diff_vs_class'] < 0].sort_values('diff_vs_class'))}\n")
by_kind = agg_features.assign(over=sig & (agg_features["diff_vs_class"] > 0), under=sig & (agg_features["diff_vs_class"] < 0)).groupby("kind").agg(
    concepts_tested=("display", "size"), significantly_over=("over", "sum"), significantly_under=("under", "sum"), median_diff_vs_class=("diff_vs_class", "median"))
print("by kind:\n" + by_kind.round(3).to_string())
agg_features